In [1]:
import pandas as pd
import re

# Load the cleaned data
df = pd.read_csv('../data/raw/complaints_raw.csv')
df = df[df['narrative'].notna()].copy()
df = df.rename(columns={'narrative': 'text'})

print(f"Starting with {len(df)} complaints with text")

C:\Users\suki\AppData\Local\Temp\ipykernel_17116\4265528030.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Starting with 162411 complaints with text


In [2]:
# Define keywords for each of YOUR logistics categories
LOGISTICS_KEYWORDS = {
    'late_delivery': [
        'never received', 'delayed', 'late', 'still waiting', 'shipping delay',
        'has not arrived', 'taking too long', 'delivery time', 'tracking',
        'package never came', 'overdue', 'awaiting delivery'
    ],
    'broken_product': [
        'broken', 'damaged', 'cracked', 'shattered', 'dented', 'crushed',
        'arrived damaged', 'package damaged', 'item damaged', 'destroyed'
    ],
    'wrong_item': [
        'wrong item', 'wrong product', 'incorrect item', 'not what i ordered',
        'received different', 'wrong size', 'wrong color', 'wrong model',
        'mistake in order', 'wrong package'
    ],
    'missing_item': [
        # Ajoute beaucoup plus de mots-clés
        'missing item', 'item missing', 'incomplete order', 'not included',
        'missing from package', 'piece missing', 'part missing', 'not in box',
        'forgot to include', 'item not in', 'short shipped', 'short by',
        'only received', 'should have received', 'expected to receive',
        'package was incomplete', 'order incomplete', 'partial order',
        'missing piece', 'missing part', 'left out', 'didn\'t include',
        'never sent', 'forgot', 'incomplete shipment'
    ],
    'poor_quality': [
        'poor quality', 'low quality', 'bad quality', 'defective',
        'cheap quality', 'not as described', 'fake', 'counterfeit'
    ],
    'transport_problem': [
        'shipping issue', 'delivery problem', 'courier', 'carrier',
        'lost in transit', 'shipping error', 'delivery driver', 'wrong address'
    ],
    'admin_error': [
        'wrong charge', 'billing error', 'wrong invoice', 'incorrect billing',
        'duplicate charge', 'wrong amount charged', 'paperwork error'
    ]
}

def categorize_complaint(text):
    """
    Looks at the complaint text and tries to find which logistics category it belongs to.
    Returns the category with the most keyword matches, or None if no match.
    """
    text = str(text).lower()
    scores = {}
    
    for category, keywords in LOGISTICS_KEYWORDS.items():
        # Count how many keywords from this category appear in the text
        score = sum(1 for kw in keywords if kw in text)
        scores[category] = score
    
    # Find the category with the highest score
    best_category = max(scores, key=scores.get)
    
    # If no keyword matched at all, return None
    if scores[best_category] == 0:
        return None
    
    return best_category

# Apply the function to all complaints (this may take 1-2 minutes)
print("Categorizing complaints... please wait")
df['logistics_category'] = df['text'].apply(categorize_complaint)

# Keep only complaints that matched a logistics category
df_logistics = df[df['logistics_category'].notna()].copy()
print(f"\nFound {len(df_logistics)} logistics-related complaints!")

# See the distribution
print("\nDistribution by category:")
print(df_logistics['logistics_category'].value_counts())

Categorizing complaints... please wait

Found 51732 logistics-related complaints!

Distribution by category:
logistics_category
late_delivery        47941
broken_product        1188
wrong_item             830
missing_item           727
poor_quality           505
transport_problem      442
admin_error             99
Name: count, dtype: int64


In [3]:
# Take a maximum of 500 examples per category to balance the dataset
MAX_PER_CATEGORY = 500

balanced_df = df_logistics.groupby('logistics_category').apply(
    lambda x: x.sample(min(len(x), MAX_PER_CATEGORY), random_state=42)
).reset_index(drop=True)

print(f"Balanced dataset size: {len(balanced_df)}")
print("\nNew distribution:")
print(balanced_df['logistics_category'].value_counts())

Balanced dataset size: 3041

New distribution:
logistics_category
broken_product       500
late_delivery        500
missing_item         500
poor_quality         500
wrong_item           500
transport_problem    442
admin_error           99
Name: count, dtype: int64


C:\Users\suki\AppData\Local\Temp\ipykernel_17116\72805154.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df_logistics.groupby('logistics_category').apply(


In [4]:
def assign_priority(row):
    """
    Assigns a priority level based on the text content and category.
    """
    text = str(row['text']).lower()
    category = row['logistics_category']
    
    # Critical keywords
    critical_words = ['urgent', 'emergency', 'medication', 'perishable', 
                      'immediately', 'critical', 'medical']
    if any(word in text for word in critical_words):
        return 'critical'
    
    # High priority
    high_words = ['frustrated', 'angry', 'unacceptable', 'lawsuit', 'lawyer',
                  'never again', 'horrible', 'terrible']
    if any(word in text for word in high_words):
        return 'high'
    
    # Category-based priority
    if category in ['broken_product', 'late_delivery']:
        return 'high'
    if category in ['poor_quality']:
        return 'low'
    
    return 'medium'

balanced_df['priority'] = balanced_df.apply(assign_priority, axis=1)
print("\nPriority distribution:")
print(balanced_df['priority'].value_counts())


Priority distribution:
priority
medium      1313
high         932
critical     399
low          397
Name: count, dtype: int64


In [5]:
import os

# Keep only the columns we need
final_en = balanced_df[['text', 'logistics_category', 'priority']].copy()
final_en.columns = ['text', 'category', 'priority']

# Create the directory if it doesn't exist
os.makedirs('./data/processed/', exist_ok=True)

# Save in English (we'll translate next)
final_en.to_csv('./data/processed/complaints_english.csv', index=False)
print(f"Saved {len(final_en)} complaints in English")

Saved 3041 complaints in English
